[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/44_dr_grpo_loss.ipynb)

# 🔴 困难：去偏组相对策略优化损失（Dr. GRPO）

实现 **Dr. GRPO（GRPO Done Right）** 损失：针对标准 DeepSeek GRPO 在 R1-Zero 类强化学习训练中存在的**回复长度膨胀偏差**（答错时故意拉长废话以摊薄每词元惩罚）和**题目难度缩放偏差**（近似全对／全错题目因极小标准差获得过大权重），精准移除了两处有偏归一化分母。

### 1. 去难度偏差的组内优势（不除以标准差）
对每个提示词组 $g$，仅减去组内奖励均值 $\bar r_g = \frac{1}{K_g}\sum_{j \in g} r_j$，**不除以**组内标准差 $\text{std}_g$：

$$
\hat A_i = r_i - \bar r_{g(i)}
$$

### 2. 词元级对称裁剪代理目标
对每个有效词元位置 $(i, t)$，计算重要性采样比率 $r_{i,t} = \exp(\text{new\_logps}_{i,t} - \text{old\_logps}_{i,t})$ 与对称裁剪代理目标：

$$
s_{i,t} = \min\Big(r_{i,t}\hat A_i,\; \operatorname{clip}(r_{i,t},\, 1 - \varepsilon,\, 1 + \varepsilon)\,\hat A_i\Big)
$$

### 3. 去长度偏差的固定常数归一化（不除以 $|o_i|$）
令固定长度归一化常数 $L_{\max}$ 为 `max_tokens`（若未指定 `max_tokens`，默认取张量词元维长度 $T = \text{new\_logps.shape}[1]$）。Dr. GRPO 对有效词元掩码求和后统一除以固定常数 $B \cdot L_{\max}$，而非每条回复自身的长度 $|o_i|$ 或动态有效词元总数：

$$
\mathcal{L}_{\text{Dr.GRPO}} = -\frac{1}{B \cdot L_{\max}} \sum_{i=1}^{B} \sum_{t=1}^{T} m_{i,t}\, s_{i,t}
$$

实现注意事项：将 `old_logps` 与优势 $\hat A_i$ 从计算图中分离，使梯度仅通过 `new_logps` 传播。

### 函数或类接口
```python
from torch import Tensor

def dr_grpo_loss(new_logps: Tensor,
                 old_logps: Tensor,
                 rewards: Tensor,
                 group_ids: Tensor,
                 mask: Tensor | None = None,
                 clip_ratio: float = 0.2,
                 max_tokens: int | None = None) -> Tensor:
    """计算一个批次的 Dr. GRPO 去偏策略梯度损失。

    new_logps: (B, T) 当前策略的词元级对数概率
    old_logps: (B, T) 旧策略（采样策略）的词元级对数概率
    rewards: (B,) 每条采样回复的标量奖励
    group_ids: (B,) 整数组编号，相同编号表示同一个提示词／组
    mask: 可选，形状为 (B, T) 的词元掩码，1 表示计入，0 表示忽略
    clip_ratio: 对称裁剪范围 eps
    max_tokens: 可选，固定长度归一化常数 L_max（为 None 时默认取 T）
    返回：标量损失（Tensor）
    """
```

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor

In [ ]:
# ✏️ 在此编写你的实现

from torch import Tensor

def dr_grpo_loss(new_logps: Tensor,
                 old_logps: Tensor,
                 rewards: Tensor,
                 group_ids: Tensor,
                 mask: Tensor | None = None,
                 clip_ratio: float = 0.2,
                 max_tokens: int | None = None) -> Tensor:
    pass  # 组内仅减均值 A_i = r_i - mean_g（不除 std）+ 词元级对称裁剪 + 统一除以固定常数 B * L_max

In [ ]:
# 🧪 调试
ratios = torch.tensor([[1.35, 1.10], [0.65, 0.90]])
old_logps = torch.zeros(2, 2)
new_logps = ratios.log().requires_grad_(True)
rewards = torch.tensor([1.0, -1.0])
group_ids = torch.tensor([0, 0])
print('Loss:', dr_grpo_loss(new_logps, old_logps, rewards, group_ids, max_tokens=2).item())

In [ ]:
# ✅ 提交
from torch_judge import check
check('dr_grpo_loss')